In [36]:
import numpy as np
import pandas as pd
import datetime

In [37]:
date = pd.read_csv('../../DataBox/orders.csv')
time = pd.read_csv('../../DataBox/messages.csv')

In [38]:
# Ensure datetime format (By Default Object) -> change data type to date time
date['date'] = pd.to_datetime(date['date'])

# --- YEAR ---
date['date_year'] = date['date'].dt.year # output -> Pandas Series
date['is_leap_year'] = date['date'].dt.is_leap_year.astype(int)

# --- MONTHS ---
date['date_month_no'] = date['date'].dt.month
date['date_month_name'] = date['date'].dt.month_name()
date['is_month_start'] = date['date'].dt.is_month_start.astype(int)
date['is_month_end'] = date['date'].dt.is_month_end.astype(int)

# --- QUARTERS & SEMESTERS ---
date['date_quarter'] = date['date'].dt.quarter
date['is_quarter_start'] = date['date'].dt.is_quarter_start.astype(int)
date['date_semester'] = np.where(date['date_quarter'].isin([1,2]), 1, 2)

# --- WEEKS ---
# .dt.week is deprecated
date['date_week'] = date['date'].dt.isocalendar().week

# --- DAYS ---
date['date_day'] = date['date'].dt.day
date['date_day_of_week'] = date['date'].dt.dayofweek
date['date_day_name'] = date['date'].dt.day_name()
date['date_day_of_year'] = date['date'].dt.dayofyear
date['date_is_weekend'] = np.where(date['date_day_name'].isin(['Sunday', 'Saturday']), 1, 0)

# --- TIME (Hours/Minutes) ---
# date['date_hour'] = date['date'].dt.hour
# date['date_minute'] = date['date'].dt.minute

# --- TIME PERIODS (Binned) ---
date['date_period'] = pd.cut(date['date'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], include_lowest=True)

# --- ELAPSED TIME (Age of record) ---
today = pd.Timestamp.today()
date['days_passed'] = (today - date['date']).dt.days
date['months_passed'] = ((today - date['date']) / np.timedelta64(1, 'm')).astype(int)


In [ ]:
# ------- DURATION CALCULATION -------

# Reference point
today = datetime.datetime.today()

# Total Timedelta
date['delta'] = today - date['date']

# Total Days (as Integer)
date['days_elapsed'] = date['delta'].dt.days

# Total Months (using average month length)
date['months_elapsed'] = np.round(date['days_elapsed'] / 30.44, 0).astype(int)

# Total Years
date['years_elapsed'] = np.round(date['days_elapsed'] / 365.25, 1)


# --- PRECISE TIME EXTRACTION ---
# Ensure the column name matches your dataframe ('time' vs 'date')
date['hour'] = date['date'].dt.hour
date['min']  = date['date'].dt.minute
date['sec']  = date['date'].dt.second

# Optional: Total seconds passed in the day (useful for some models)
date['seconds_since_midnight'] = (date['hour'] * 3600) + (date['min'] * 60) + date['sec']

In [ ]:
# Extract Time part
time['date'] = pd.to_datetime(time['date'], errors='coerce') # Object to date time
time['time'] = time['date'].dt.time

# Total Seconds (The most precise numeric representation)
time['total_seconds'] = (time['date'].dt.hour * 3600) + (time['date'].dt.minute * 60) + (time['date'].dt.second)

# Total Minutes (Good for mid-range granularity)
time['total_minutes'] = (time['date'].dt.hour * 60) + time['date'].dt.minute

# Time Partitions (Binned)
time['time_of_day'] = pd.cut(time['date'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

# AM/PM Indicator
time['is_am'] = np.where(time['date'].dt.hour < 12, 1, 0)

In [ ]:
# Ensure date is converted (prevents .dt accessor errors)
time['date'] = pd.to_datetime(time['date'], errors='coerce')
today = datetime.datetime.today()

# Raw Time Difference (Timedelta object)
time['diff'] = today - time['date']

# Difference in SECONDS
time['diff_seconds'] = time['diff'] / np.timedelta64(1, 's')

# Difference in MINUTES
time['diff_minutes'] = time['diff'] / np.timedelta64(1, 'm')

# Difference in HOURS
time['diff_hours'] = time['diff'] / np.timedelta64(1, 'h')

# Difference in DAYS (Standard alternative)
time['diff_days'] = time['diff'].dt.days